# Session 13 — 3/3: push the batch, scan depth, build the report

Run 3 takes the winning reversible variant and pushes the batch size to the
edge of the GPU. Then a depth scan isolates the actual claim of reversibility:
activation memory that does not grow with the number of layers.

In [ ]:
!nvidia-smi
!git clone https://github.com/rjvim/era-v5-session13-reversibility repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, 'src')

## Max batch for each mode

In [ ]:
!python src/train.py --mode midpoint --find_max_batch --seq_len 512
!python src/train.py --mode euler    --find_max_batch --seq_len 512

## Run 3 — midpoint @ max batch

In [ ]:
import json
BATCH_MAX = json.load(open('results/maxbatch_midpoint.json'))['max_batch']
BATCH_FIX = json.load(open('results/maxbatch_baseline.json'))['max_batch']
print(f'baseline max {BATCH_FIX} -> midpoint max {BATCH_MAX} ({BATCH_MAX/BATCH_FIX:.2f}x)')
!python src/train.py --mode midpoint --batch_size {BATCH_MAX} --run_name midpoint_max     --h 0.5 --check_recon --seq_len 512 --total_tokens 50000000 --resume


## Memory vs depth

Same batch and sequence length, layers swept. Baseline should grow roughly linearly; midpoint should stay flat.

In [ ]:
!python scripts/depth_scan.py --layers 4 8 16 32 64 --batch_size 8 --seq_len 512

## Cost comparison

What the memory saving is worth in rupees/dollars: slower per token, but a smaller GPU tier or fewer nodes.

In [ ]:
import json
b = json.load(open('results/baseline_fixed.json'))
m = json.load(open('results/midpoint_fixed.json'))
PRICE_PER_HR = 0.35   # <-- set to the actual price of the GPU tier you used
for name, d in [('baseline', b), ('midpoint', m)]:
    hrs = d['tokens_trained'] / d['tok_per_s'] / 3600
    print(f"{name:9s} {hrs:.2f} GPU-hours  ${hrs*PRICE_PER_HR:.2f} for "
          f"{d['tokens_trained']:,} tokens  (peak {d['peak_mem_gb']:.2f} GB)")
print('\nreversibility costs %.1f%% more compute-time for %.2fx the peak memory'
      % (100*(b['tok_per_s']/m['tok_per_s']-1), m['peak_mem_gb']/b['peak_mem_gb']))

## Generate the README and verify it

In [ ]:
!python scripts/make_readme.py
!pytest tests/ -q

In [ ]:
!git add -A
!git -c user.email=rajivs.iitkgp@gmail.com -c user.name=rjvim commit -q -m 'session 13: results + generated README'
!git push
